# Metales Pesados Industriales (RETC) — Santiago RM

**Objetivo:** Construir una capa comunal de emisiones industriales de metales pesados
neurotóxicos (Pb, As, Hg, Mn, Cd) usando el Registro de Emisiones y Transferencias
de Contaminantes (RETC) del Ministerio del Medio Ambiente de Chile.

## Relevancia para salud cerebral

La **Lancet Commission on Dementia 2024** identifica la contaminación del aire
como el factor modificable #1 de riesgo de demencia a nivel poblacional. El Plomo (Pb)
tiene la evidencia más sólida entre los metales pesados:

- Reduce el volumen hipocampal de forma dosis-respuesta (Lanphear et al., *Lancet Public Health* 2018).
- Su relación exposición-cognición es **log-lineal**: los mayores efectos se dan a bajas concentraciones.
- Las fuentes industriales puntuales (fundición, reciclaje de baterías) generan
  gradientes de exposición que **no co-localizan** con la contaminación de fondo por
  combustión (PM₂.₅), representando un pathway exposómico independiente.

**Fuente:** RETC MMA — `datosretc.mma.gob.cl`, CC-BY, sin API key.
Cobertura: 2015–2022 (CSV 2015–2020 + XLSX 2021–2022).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "processed"
FIGURES_DIR = REPO_ROOT / "figures"

plt.rcParams.update({"figure.dpi": 110, "font.family": "STIXGeneral"})

HM_CSV = DATA_DIR / "santiago_heavy_metals_retc_2015_2022.csv"
MASTER_GEO = DATA_DIR / "santiago_exposome_master.geojson"

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Heavy metals CSV exists: {HM_CSV.exists()}")

## 1. Carga de datos

El archivo `santiago_heavy_metals_retc_2015_2022.csv` fue generado por
`scripts/run_heavy_metals.py` (fuente: `src/exposome/heavy_metals.py`).
Una fila por comuna, sin filas duplicadas ni missing.

In [ ]:
hm = pd.read_csv(HM_CSV)
print(f"Shape: {hm.shape}")
print(f"Columns: {hm.columns.tolist()}")
hm.head()

## 2. Estadísticos descriptivos — Tiltil como outlier

La distribución de Pb es extremadamente sesgada: Tiltil concentra casi toda
la emisión industrial del RM. La transformación log₁p es necesaria para
cualquier modelo estadístico.

In [ ]:
# Estadísticos básicos por metal
metal_cols = ["hm_pb_kg", "hm_mn_kg", "hm_as_kg", "hm_cd_kg", "hm_hg_kg"]
print("=== Emisiones medias anuales por metal [kg/yr] ===")
print(hm[metal_cols].describe().round(2))

print("\n=== Top 5 comunas por Pb ===")
print(hm.nlargest(5, "hm_pb_kg")[["name", "hm_pb_kg", "hm_pb_log", "n_sources"]].to_string(index=False))

print("\n=== Comunas con n_sources > 0 ===")
print(f"N comunas con fuentes declaradas: {(hm['n_sources'] > 0).sum()} / {len(hm)}")

# Confirmación de Mn y Cd = 0 en RM
print(f"\nMn total RM: {hm['hm_mn_kg'].sum():.1f} kg/yr (esperado 0 — sin fuentes industriales en RM)")
print(f"Cd total RM: {hm['hm_cd_kg'].sum():.1f} kg/yr (esperado 0 — sin fuentes industriales en RM)")

## 3. Figura de 4 paneles

La figura generada por `scripts/plot_heavy_metals_map.py` muestra:
- **A)** Coropleta log(Pb+1) sobre el mapa comunal
- **B)** Ranking de las 20 comunas con más emisiones de Pb
- **C)** Correlación Pb vs PM₂.₅ (¿co-localización de exposiciones?)
- **D)** Gradiente socioeconómico de las emisiones

In [ ]:
import subprocess
result = subprocess.run(
    ["python", str(REPO_ROOT / "scripts" / "plot_heavy_metals_map.py")],
    capture_output=True, text=True, cwd=REPO_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

from IPython.display import Image
fig_path = FIGURES_DIR / "heavy_metals_santiago_4panel.png"
if fig_path.exists():
    display(Image(str(fig_path), width=900))

## 4. Validez de constructo — correlaciones con el master

Cargamos el master completo y calculamos correlaciones de Spearman entre
la capa de metales pesados y otras exposiciones del exposome.

In [ ]:
gdf = gpd.read_file(MASTER_GEO)

pb_col = "hm_pb_log"

comparisons = {
    "PM₂.₅ crónico (ACAG)": "pm25_pop_weighted",
    "NO₂ superficie (Sentinel-5P)": "no2_surface_ugm3",
    "ALAN (VIIRS)": "alan_pop_weighted_mean",
    "Índice NSE (PCA)": "nse_index_pca",
    "% Pobreza (CASEN)": "pobreza_p",
    "Índice de incendios": "fire_index",
}

print(f"Correlaciones Spearman: log(Pb+1) vs otras exposiciones (n={len(gdf)})")
print(f"{'Variable':<35} {'ρ':>6} {'p':>8} {'sig':>5}")
print("-" * 60)

for label, col in comparisons.items():
    if col in gdf.columns:
        valid = gdf[[pb_col, col]].dropna()
        rho, pval = spearmanr(valid[pb_col], valid[col])
        sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "n.s."))
        print(f"{label:<35} {rho:>+.3f} {pval:>8.3f} {sig:>5}")
    else:
        print(f"{label:<35} [columna no encontrada en master]")

## 5. Metodología resumida

### Fuente
- **RETC** (Registro de Emisiones y Transferencias de Contaminantes),
  Ministerio del Medio Ambiente. Dataset `2733b0f0-...`. CC-BY. Sin API key.
- Años: 2015–2020 (CSV semicolón, latin-1/utf-8-sig) y 2021–2022 (XLSX).

### Procesamiento (`src/exposome/heavy_metals.py`)
1. **Descarga y caché** por año en `cache/retc_efp_YYYY.*`.
2. **Filtro RM**: filas con `region` que contenga "Metropolitana".
3. **Clasificación de metales**: búsqueda por palabras clave en español
   (`plomo`, `arsénico`, `cadmio`, `manganeso`, `mercurio`).
4. **Conversión de unidades**: ton/año → kg × 1000; g/año → kg / 1000.
5. **Geocodificación**: `latitud`/`longitud` en formato español (coma decimal);
   punto-en-polígono con fallback `sjoin_nearest` en EPSG:32719.
6. **Agregación**: media anual por comuna (2015–2022).
7. **Derivados**: `hm_pb_log = log1p(hm_pb_kg)`, `hm_as_log`, `hm_index`
   (z-score ponderado Pb×0.35 + Mn×0.30 + As×0.20 + Cd×0.10 + Hg×0.05).

### Hallazgos clave
- **Tiltil outlier**: ~10,629 kg/yr Pb ≈ 99.6% del total RM. Complejo industrial conocido.
- **Mn y Cd = 0 en RM**: hallazgo real (no error). Sus fuentes industriales están en otras regiones.
- **Pb vs PM₂.₅**: ρ ≈ −0.15 (p ≈ 0.30, n.s.) → **pathways independientes**.

Metodología completa: `docs/heavy_metals_methodology.md`.

## 6. Confirmación de integración en el master

In [ ]:
hm_cols_in_master = [c for c in gdf.columns if c.startswith("hm_")]
print(f"Columnas de metales pesados en el master: {hm_cols_in_master}")
print(f"Total columnas en master: {len(gdf.columns)}")
print(f"Filas (comunas): {len(gdf)}")
print(f"\nResumen hm_pb_log en master:")
print(gdf["hm_pb_log"].describe().round(3))